<a href="https://colab.research.google.com/github/Joyanne05/vres-sentiment-analysis/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [9]:
df = pd.read_csv("course_data_clean.csv")
df = df[["reviews", "course_rating_int"]]
df.head()

,reviews,course_rating_int
0,go to office hours and practice,1.0
1,One of my least favourite courses. Although th...,0.0
2,It starts with a very low pace but after midte...,0.0
3,Took this in 2018 with no programming experien...,1.0
4,I loved everything about cs 115. Great instruc...,1.0


In [11]:
print(df["course_rating_int"].value_counts(dropna=False))

course_rating_int
1.0    9786
0.0    4830
NaN     222
Name: count, dtype: int64


In [13]:
# Remove rows missing a review or rating
df = df.dropna(subset=["reviews", "course_rating_int"]).copy()

# Remove empty or whitespace-only reviews
df = df[df["reviews"].str.strip().ne("")].copy()

# Convert labels to integer
df["course_rating_int"] = df["course_rating_int"].astype(int)

# Check the result
print(df["course_rating_int"].value_counts(dropna=False))
print("Remaining rows:", len(df))

course_rating_int
1    9784
0    4824
Name: count, dtype: int64
Remaining rows: 14608


In [14]:
# Split dataset (80/20)
X = df["reviews"]
y = df["course_rating_int"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [27]:
# Vectorisation using TF-IDF
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training matrix:", X_train_tfidf.shape)
print("Test matrix:", X_test_tfidf.shape)


Training matrix: (11686, 12366)
Test matrix: (2922, 12366)


In [26]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [28]:
y_pred = lr_model.predict(X_test_tfidf)

In [29]:
from sklearn.metrics import accuracy_score, classification_report

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")

print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Negative", "Positive"],
    zero_division=0
))

Accuracy: 0.776
              precision    recall  f1-score   support

    Negative       0.76      0.47      0.58       965
    Positive       0.78      0.93      0.85      1957

    accuracy                           0.78      2922
   macro avg       0.77      0.70      0.71      2922
weighted avg       0.77      0.78      0.76      2922



In [30]:
# Multinomial Naive Bayes
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

nb_pred = nb_model.predict(X_test_tfidf)

In [31]:
from sklearn.metrics import accuracy_score, classification_report

print(f"Accuracy: {accuracy_score(y_test, nb_pred):.3f}")

print(classification_report(
    y_test,
    nb_pred,
    labels=[0, 1],
    target_names=["Negative", "Positive"],
    zero_division=0
))

Accuracy: 0.708
              precision    recall  f1-score   support

    Negative       0.96      0.12      0.22       965
    Positive       0.70      1.00      0.82      1957

    accuracy                           0.71      2922
   macro avg       0.83      0.56      0.52      2922
weighted avg       0.78      0.71      0.62      2922



In [32]:
# Support Vector Machine
from sklearn.svm import LinearSVC

svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)

svm_pred = svm_model.predict(X_test_tfidf)

print(f"Accuracy: {accuracy_score(y_test, svm_pred):.3f}")

print(classification_report(
    y_test,
    svm_pred,
    labels=[0, 1],
    target_names=["Negative", "Positive"],
    zero_division=0
))

Accuracy: 0.765
              precision    recall  f1-score   support

    Negative       0.68      0.55      0.61       965
    Positive       0.80      0.87      0.83      1957

    accuracy                           0.76      2922
   macro avg       0.74      0.71      0.72      2922
weighted avg       0.76      0.76      0.76      2922



In [35]:
import pandas as pd

baseline_results = pd.DataFrame({
    "model": ["Logistic Regression", "Multinomial Naive Bayes", "Linear SVM"],
    "accuracy": [0.776, 0.708, 0.765],
    "macro_f1": [0.71, 0.52, 0.72],
    "negative_recall": [0.47, 0.12, 0.55]
})

baseline_results.to_csv("baseline_results.csv", index=False)
baseline_results

,model,accuracy,macro_f1,negative_recall
0,Logistic Regression,0.776,0.71,0.47
1,Multinomial Naive Bayes,0.708,0.52,0.12
2,Linear SVM,0.765,0.72,0.55


## Cross validation

In [36]:
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import make_scorer, recall_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [37]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(random_state=42)
}

scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    "negative_recall": make_scorer(
        recall_score,
        pos_label=0,
        zero_division=0
    )
}

In [41]:
cv_results = {}
summary = []

for name, model in models.items():

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        error_score="raise"
    )

    cv_results[name] = scores

    summary.append({
        "model": name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "macro_f1_mean": scores["test_macro_f1"].mean(),
        "macro_f1_std": scores["test_macro_f1"].std(),
        "negative_recall_mean": scores["test_negative_recall"].mean()
    })

cv_summary = pd.DataFrame(summary)
print("5-fold cross-validation with fixed model settings")
display(cv_summary.round(3))

5-fold cross-validation with fixed model settings


,model,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,negative_recall_mean
0,Logistic Regression,0.785,0.010,0.729,0.012,0.500
1,Naive Bayes,0.707,0.003,0.518,0.009,0.123
2,Linear SVM,0.773,0.007,0.732,0.008,0.574


In [39]:
from sklearn.model_selection import GridSearchCV

param_grids = {
    "Logistic Regression": {
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "clf__C": [0.1, 1, 10],
        "clf__class_weight": [None, "balanced"]
    },

    "Naive Bayes": {
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "clf__alpha": [0.1, 0.5, 1, 2]
    },

    "Linear SVM": {
        "tfidf__ngram_range": [(1, 1), (1, 2)],
        "clf__C": [0.1, 1, 10],
        "clf__class_weight": [None, "balanced"]
    }
}

In [40]:
searches = {}
tuning_summary = []

for name, model in models.items():

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf", model)
    ])

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[name],
        scoring=scoring,
        refit="macro_f1",
        cv=cv,
        n_jobs=-1,
        verbose=1,
        error_score="raise"
    )

    search.fit(X_train, y_train)
    searches[name] = search

    best_index = search.best_index_
    results = search.cv_results_

    tuning_summary.append({
        "model": name,
        "cv_accuracy": results["mean_test_accuracy"][best_index],
        "cv_macro_f1": results["mean_test_macro_f1"][best_index],
        "cv_macro_f1_std": results["std_test_macro_f1"][best_index],
        "cv_negative_recall": results["mean_test_negative_recall"][best_index],
        "best_params": search.best_params_
    })

    print(f"\n{name}")
    print("Best settings:", search.best_params_)
    print(f"Best CV macro-F1: {search.best_score_:.3f}")

tuned_results = pd.DataFrame(tuning_summary)

display(tuned_results.drop(columns="best_params").round(3))

Fitting 5 folds for each of 12 candidates, totalling 60 fits

Logistic Regression
Best settings: {'clf__C': 10, 'clf__class_weight': 'balanced', 'tfidf__ngram_range': (1, 2)}
Best CV macro-F1: 0.758
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Naive Bayes
Best settings: {'clf__alpha': 0.1, 'tfidf__ngram_range': (1, 1)}
Best CV macro-F1: 0.672
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Linear SVM
Best settings: {'clf__C': 1, 'clf__class_weight': 'balanced', 'tfidf__ngram_range': (1, 2)}
Best CV macro-F1: 0.755


,model,cv_accuracy,cv_macro_f1,cv_macro_f1_std,cv_negative_recall
0,Logistic Regression,0.788,0.758,0.007,0.663
1,Naive Bayes,0.757,0.672,0.010,0.377
2,Linear SVM,0.786,0.755,0.007,0.652
